In [23]:
import pandas as pd
from collections import Counter

In [41]:
%run data_preprocessing.ipynb

In [25]:
def each_journey_length(data, target_column='user_journey'):
    """
    Calculate journey length statistics - number of pages per user journey.
    
    Args:
        data (pd.DataFrame): Preprocessed DataFrame with grouped user journeys
        target_column (str): Column containing journey strings
        
    Returns:
        pd.DataFrame: Statistics about journey lengths
    """
    # Count pages in each journey
    data_copy = data.copy()
    data_copy['journey_length'] = data_copy[target_column].apply(
        lambda x: len(x.split('-')) if pd.notna(x) else 0
    )
    
    # Calculate statistics
    stats = {
        'avg_length': data_copy['journey_length'].mean(),
        'min_length': data_copy['journey_length'].min(),
        'max_length': data_copy['journey_length'].max(),
        'median_length': data_copy['journey_length'].median(),
        'total_users': len(data_copy)
    }
    
    result = pd.DataFrame([stats])
    return result


In [26]:
def each_page_count_in_all_users(data, target_column='user_journey'):
    """
    Count how many times each page appears across all user journeys.
    
    Args:
        data (pd.DataFrame): Preprocessed DataFrame with grouped user journeys
        target_column (str): Column containing journey strings
        
    Returns:
        pd.DataFrame: Pages sorted by frequency (most common first)
    """
    # Count all page occurrences
    page_counts = Counter()
    
    for journey in data[target_column].dropna():
        pages = journey.split('-')
        page_counts.update(pages)
    
    # Convert to DataFrame
    result = pd.DataFrame([
        {'page': page, 'count': count} 
        for page, count in page_counts.items()
    ])
    
    # Sort by count (descending)
    result = result.sort_values('count', ascending=False).reset_index(drop=True)
    
    return result


In [29]:
def each_page_count_per_user_journey(data, target_column='user_journey'):
    """
    Count how many unique journeys contain each page (counts each page once per journey).
    
    Args:
        data (pd.DataFrame): Preprocessed DataFrame with grouped user journeys
        target_column (str): Column containing journey strings
        
    Returns:
        pd.DataFrame: Pages sorted by presence (most common first)
    """
    # Count unique journeys containing each page
    page_presence = Counter()
    
    for journey in data[target_column].dropna():
        # Get unique pages in this journey
        unique_pages = set(journey.split('-'))
        page_presence.update(unique_pages)
    
    # Convert to DataFrame
    result = pd.DataFrame([
        {'page': page, 'journey_count': count, 'percentage': (count / len(data)) * 100} 
        for page, count in page_presence.items()
    ])
    
    # Sort by journey_count (descending)
    result = result.sort_values('journey_count', ascending=False).reset_index(drop=True)
    
    return result


In [31]:
def landing_page_after_each_page(data, target_column='user_journey', top_n=5):
    """
    Show which pages follow after each page (transition analysis).
    
    Args:
        data (pd.DataFrame): Preprocessed DataFrame with grouped user journeys
        target_column (str): Column containing journey strings
        top_n (int): Number of top destinations to show per page
        
    Returns:
        dict: Dictionary mapping each page to its top destinations
    """
    # Track transitions: page -> next_page
    transitions = {}
    
    for journey in data[target_column].dropna():
        pages = journey.split('-')
        
        # Look at each consecutive pair
        for i in range(len(pages) - 1):
            current_page = pages[i]
            next_page = pages[i + 1]
            
            if current_page not in transitions:
                transitions[current_page] = Counter()
            
            transitions[current_page][next_page] += 1
    
    # Format results - get top N destinations for each page
    result = {}
    for page, destinations in transitions.items():
        top_destinations = destinations.most_common(top_n)
        result[page] = [
            {'destination': dest, 'count': count} 
            for dest, count in top_destinations
        ]
    
    return result


In [34]:
def most_frequent_page_sequences(data, target_column='user_journey', sequence_length=4, top_n=10):
    """
    Find the most popular sequences of N consecutive pages.
    
    Args:
        data (pd.DataFrame): Preprocessed DataFrame with grouped user journeys
        target_column (str): Column containing journey strings
        sequence_length (int): Length of sequences to find (default: 3)
        top_n (int): Number of top sequences to return
        
    Returns:
        pd.DataFrame: Top N most common page sequences
    """
    # Count all sequences of the specified length
    sequence_counts = Counter()
    
    for journey in data[target_column].dropna():
        pages = journey.split('-')
        
        # Extract all sequences of length N
        for i in range(len(pages) - sequence_length + 1):
            sequence = '-'.join(pages[i:i + sequence_length])
            sequence_counts[sequence] += 1
    
    # Get top N sequences
    top_sequences = sequence_counts.most_common(top_n)
    
    # Convert to DataFrame
    result = pd.DataFrame([
        {'sequence': seq, 'count': count, 'rank': idx + 1} 
        for idx, (seq, count) in enumerate(top_sequences)
    ])
    
    return result


In [ ]:
if __name__ == "__main__":
    # Load data
    file_path = '../data/user_journey_raw.csv'
    data = pd.read_csv(file_path)
    
    # group all the sessions based on the user and concatenate them
    grouped_data = group_by(data, sessions='3')

    #print(grouped_data)
    
    # Clean the user journeys
    cleaned_data = remove_page_duplicates(grouped_data, 'user_journey')
    #print(cleaned_data)
    journey_length = each_journey_length(cleaned_data)
    #print(journey_length)
    page_count = each_page_count_in_all_users(cleaned_data)
    #print(page_count)
    page_presence_count = each_page_count_per_user_journey(cleaned_data)
    #print(page_presence_count)
    page_destination = landing_page_after_each_page(cleaned_data)
    #print(page_destination)
    page_sequence = most_frequent_page_sequences(cleaned_data)
    #print(page_sequence)

In [ ]:
# questions
# How many records are in the data if you group only the first three sessions?
